# Replication: Bhattacharya, Wilson & Soyer (2019)

**"A Bayesian approach to modeling mortgage default and prepayment"**  
*European Journal of Operational Research*, 274, 1112–1124.

---

## Paper Overview

This notebook replicates the Bayesian competing risks proportional hazards model
from Bhattacharya et al. (2019). The paper models mortgage termination as a
**competing risks** problem where each loan faces two mutually exclusive risks:

| Risk | Description |
|------|-------------|
| **Default** ($T_D$) | Borrower fails to meet mortgage obligations |
| **Prepayment** ($T_P$) | Borrower pays off the mortgage early |

The observed termination time is $T_M = \min(T_D, T_P, C)$ where $C$ is the
right-censoring time.

### Model Specification (Paper Section 2)

The cause-specific hazard functions follow a **proportional hazards** form:

$$\lambda_k(t \mid \mathbf{x}) = r_k(t \mid \mu_k, \sigma_k) \exp(\boldsymbol{\theta}_k' \mathbf{x}), \quad k \in \{D, P\}$$

where $r_k(t)$ is a **lognormal** baseline hazard:

$$r_k(t \mid \mu_k, \sigma_k) = \frac{\phi(z_k)}{\sigma_k \cdot t \cdot [1 - \Phi(z_k)]}, \quad z_k = \frac{\log t - \mu_k}{\sigma_k}$$

### Priors (Paper Section 3)

| Parameter | Prior |
|-----------|-------|
| $\boldsymbol{\theta}_D, \boldsymbol{\theta}_P$ | $\mathcal{N}(\mathbf{0}, 100^2 \mathbf{I})$ |
| $\mu_D, \mu_P$ | $\mathcal{N}(0, 10^2)$ |
| $\sigma_D, \sigma_P$ | $\text{Exp}(0.01)$ &mdash; mean = 100 |

### Data

Freddie Mac 1999 vintage, single-family 30-year fixed-rate loans from the
Single-Family Loan-Level Dataset (sample files). The paper uses 12 covariates
plus an intercept.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')

# Loan origination node
ax.add_patch(mpatches.FancyBboxPatch((0.5, 2.2), 2.2, 1.2, boxstyle='round,pad=0.2',
                                      facecolor='#4a90d9', edgecolor='black', linewidth=1.5))
ax.text(1.6, 2.8, 'Loan\nOrigination', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Observation period
ax.annotate('', xy=(3.8, 2.8), xytext=(2.9, 2.8),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(3.35, 3.15, '$T_M$', ha='center', va='bottom', fontsize=13, fontstyle='italic')

# Competing risks node
ax.add_patch(mpatches.FancyBboxPatch((3.8, 2.2), 2.2, 1.2, boxstyle='round,pad=0.2',
                                      facecolor='#f5a623', edgecolor='black', linewidth=1.5))
ax.text(4.9, 2.8, 'Termination\nObserved', ha='center', va='center', fontsize=11, fontweight='bold')

# Default branch
ax.annotate('', xy=(8.0, 4.8), xytext=(6.2, 3.2),
            arrowprops=dict(arrowstyle='->', lw=2, color='#d0021b'))
ax.add_patch(mpatches.FancyBboxPatch((7.2, 4.2), 2.2, 1.2, boxstyle='round,pad=0.2',
                                      facecolor='#d0021b', edgecolor='black', linewidth=1.5))
ax.text(8.3, 4.8, 'Default\n$T_D$', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Prepayment branch
ax.annotate('', xy=(8.0, 2.8), xytext=(6.2, 2.8),
            arrowprops=dict(arrowstyle='->', lw=2, color='#7ed321'))
ax.add_patch(mpatches.FancyBboxPatch((7.2, 2.2), 2.2, 1.2, boxstyle='round,pad=0.2',
                                      facecolor='#7ed321', edgecolor='black', linewidth=1.5))
ax.text(8.3, 2.8, 'Prepayment\n$T_P$', ha='center', va='center', fontsize=11, fontweight='bold')

# Censored branch
ax.annotate('', xy=(8.0, 0.8), xytext=(6.2, 2.4),
            arrowprops=dict(arrowstyle='->', lw=2, color='#9b9b9b'))
ax.add_patch(mpatches.FancyBboxPatch((7.2, 0.2), 2.2, 1.2, boxstyle='round,pad=0.2',
                                      facecolor='#9b9b9b', edgecolor='black', linewidth=1.5))
ax.text(8.3, 0.8, 'Censored\n$C$', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.set_title('Figure 1: Competing Risks Framework for Mortgage Termination', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

---

## Section 2: Data Loading

Load the raw 1999 vintage origination and monthly performance files from the
Freddie Mac Single-Family Loan-Level Dataset (sample).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Bayesian inference
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import arviz as az

# ML comparison
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Project imports
import sys
sys.path.insert(0, '../src')
from data.columns import (
    ORIGINATION_COLUMNS, PERFORMANCE_COLUMNS,
    ORIGINATION_DTYPES, PERFORMANCE_DTYPES,
    MISSING_VALUES, MATURITY_THRESHOLD_MONTHS,
)
from data.preprocess import load_origination_data, load_performance_data

sns.set_style('whitegrid')
%matplotlib inline

np.random.seed(42)
pyro.set_rng_seed(42)

# Paths
DATA_DIR = Path('../data/raw/sample_1999')
FIGURES_DIR = Path('../reports/figures')
MODELS_DIR = Path('../models')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Device
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'PyTorch {torch.__version__}, Pyro {pyro.__version__}')
print(f'Device: {DEVICE}')

In [ ]:
# Load origination data
orig_df = load_origination_data(DATA_DIR / 'sample_orig_1999.txt')
print(f'Origination records: {len(orig_df):,}')
print(f'Columns: {orig_df.shape[1]}')
orig_df.head(2)

In [ ]:
# Load monthly performance data
perf_df = load_performance_data(DATA_DIR / 'sample_svcg_1999.txt')
print(f'Performance records: {len(perf_df):,}')
print(f'Unique loans: {perf_df["loan_sequence_number"].nunique():,}')
perf_df.head(2)

---

## Section 3: Loan Categorization (Paper Section 4.1)

Following the paper’s exact categorization:
- **Prepaid**: `zero_balance_code == '01'` and not near maturity
- **Default**: `zero_balance_code` in `{'03', '06', '09'}` (short sale, REO, etc.)
- **Active/Censored**: No terminal event observed

Duration is measured in **years** (paper’s Figure 2 x-axis spans 0–15 years).

In [ ]:
# Sort performance data chronologically
perf_df = perf_df.sort_values(['loan_sequence_number', 'loan_age'])

# Find first terminal event per loan
terminal_codes_prepay = {'01'}
terminal_codes_default = {'03', '06', '09'}
terminal_codes_all = terminal_codes_prepay | terminal_codes_default

# Get rows where a terminal event occurred
perf_df['zbc_clean'] = perf_df['zero_balance_code'].fillna('').astype(str).str.strip().str.zfill(2)
terminal_rows = perf_df[perf_df['zbc_clean'].isin(terminal_codes_all)].copy()

# Keep first terminal event per loan
first_terminal = terminal_rows.groupby('loan_sequence_number').first().reset_index()
first_terminal = first_terminal[['loan_sequence_number', 'loan_age', 'zbc_clean']].copy()
first_terminal.columns = ['loan_sequence_number', 'terminal_loan_age', 'terminal_code']

# Get last observation per loan (for censored loans)
last_obs = perf_df.groupby('loan_sequence_number').agg(
    last_loan_age=('loan_age', 'last'),
    last_reporting=('monthly_reporting_period', 'last'),
).reset_index()

# Merge origination info for maturity check
loan_info = orig_df[['loan_sequence_number', 'orig_loan_term']].copy()

# Build loan-level dataset
loans = last_obs.merge(loan_info, on='loan_sequence_number', how='left')
loans = loans.merge(first_terminal, on='loan_sequence_number', how='left')

# Classify each loan
def classify_loan(row):
    if pd.isna(row['terminal_code']) or row['terminal_code'] == '00':
        return 'active'
    if row['terminal_code'] in terminal_codes_default:
        return 'default'
    if row['terminal_code'] in terminal_codes_prepay:
        # Check maturity: if loan_age near orig_loan_term, it matured (not prepaid)
        age = row['terminal_loan_age']
        term = row['orig_loan_term']
        if pd.notna(age) and pd.notna(term) and age >= term - MATURITY_THRESHOLD_MONTHS:
            return 'active'  # Matured loans treated as censored
        return 'prepaid'
    return 'active'

loans['category'] = loans.apply(classify_loan, axis=1)

# Compute duration in YEARS
loans['duration_months'] = np.where(
    loans['category'].isin(['default', 'prepaid']),
    loans['terminal_loan_age'],
    loans['last_loan_age']
)
loans['duration_years'] = loans['duration_months'] / 12.0

# Map event codes: 0=censored, 1=prepay, 2=default
event_map = {'active': 0, 'prepaid': 1, 'default': 2}
loans['event_code'] = loans['category'].map(event_map)

# Drop loans with missing duration
loans = loans.dropna(subset=['duration_months'])
loans = loans[loans['duration_months'] > 0].copy()

print(f'\nLoan categorization:')
for cat in ['prepaid', 'default', 'active']:
    n = (loans['category'] == cat).sum()
    print(f'  {cat:10s}: {n:6,} ({n/len(loans)*100:5.1f}%)')
print(f'  {"Total":10s}: {len(loans):6,}')

---

## Section 4: Covariate Engineering (Paper Section 4.2)

The paper uses **12 covariates + intercept**. Quantitative covariates are
standardized to zero mean and unit variance. Binary indicators are used for
categorical variables.

Dropped per paper: CLTV (correlated with MI%), current interest rate (correlated
with original interest rate).

In [ ]:
# Judicial vs non-judicial foreclosure states (paper re-categorization)
JUDICIAL_STATES = {
    'CT', 'DE', 'FL', 'HI', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA',
    'ME', 'MD', 'MA', 'MI', 'MN', 'NE', 'NJ', 'NM', 'NY', 'NC',
    'ND', 'OH', 'OK', 'PA', 'SC', 'SD', 'VT', 'WI'
}

# Merge origination covariates onto loan-level data
orig_covars = orig_df[[
    'loan_sequence_number', 'credit_score', 'mi_pct', 'num_units',
    'orig_dti', 'orig_upb', 'orig_interest_rate', 'num_borrowers',
    'first_time_homebuyer', 'occupancy_status', 'property_state',
    'property_type',
]].copy()

df = loans.merge(orig_covars, on='loan_sequence_number', how='left')

# --- Handle missing value codes ---
for col, codes in MISSING_VALUES.items():
    if col in df.columns:
        df[col] = df[col].replace(codes, np.nan)

# Convert to numeric
for col in ['credit_score', 'mi_pct', 'num_units', 'orig_dti', 'orig_upb',
            'orig_interest_rate', 'num_borrowers']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --- Binary indicators ---

# First-time homebuyer: Y=1, else=0
df['first_time_homebuyer_ind'] = (df['first_time_homebuyer'] == 'Y').astype(float)

# Occupancy status: non-owner-occupied=1 (investment/second home), owner-occupied=0
df['occupancy_non_owner'] = df['occupancy_status'].apply(
    lambda x: 1.0 if x in ('I', 'S') else 0.0
)

# Foreclosure state: judicial=1, non-judicial=0
df['foreclosure_judicial'] = df['property_state'].apply(
    lambda x: 1.0 if x in JUDICIAL_STATES else 0.0
)

# Property type: non-single-family=1, single-family=0
df['property_non_sf'] = df['property_type'].apply(
    lambda x: 0.0 if x == 'SF' else 1.0
)

# Fill mi_pct NaN with 0 (no mortgage insurance)
df['mi_pct'] = df['mi_pct'].fillna(0)

# --- Define covariate lists ---
QUANTITATIVE_COVARIATES = [
    'credit_score', 'mi_pct', 'num_units', 'orig_dti',
    'orig_upb', 'orig_interest_rate', 'num_borrowers',
]

BINARY_COVARIATES = [
    'first_time_homebuyer_ind', 'occupancy_non_owner',
    'foreclosure_judicial', 'property_non_sf',
]

ALL_COVARIATES = QUANTITATIVE_COVARIATES + BINARY_COVARIATES

# Drop rows with missing covariates
n_before = len(df)
df = df.dropna(subset=QUANTITATIVE_COVARIATES).copy()
print(f'Dropped {n_before - len(df):,} rows with missing covariates ({(n_before-len(df))/n_before*100:.1f}%)')
print(f'Final dataset: {len(df):,} loans')

# Category distribution after cleaning
print(f'\nCategory distribution after cleaning:')
for cat in ['prepaid', 'default', 'active']:
    n = (df['category'] == cat).sum()
    print(f'  {cat:10s}: {n:6,} ({n/len(df)*100:5.1f}%)')

---

## Section 5: Exploratory Data Analysis

In [ ]:
# Figure 2: Histogram of time to event by category
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'prepaid': '#4a90d9', 'default': '#d0021b', 'active': '#9b9b9b'}
labels = {'prepaid': f'Prepaid (n={int((df["category"]=="prepaid").sum()):,})',
          'default': f'Default (n={int((df["category"]=="default").sum()):,})',
          'active':  f'Active/Censored (n={int((df["category"]=="active").sum()):,})'}

bins = np.linspace(0, 16, 65)
for cat in ['prepaid', 'default', 'active']:
    mask = df['category'] == cat
    ax.hist(df.loc[mask, 'duration_years'], bins=bins, alpha=0.6,
            color=colors[cat], label=labels[cat], edgecolor='white', linewidth=0.3)

ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Figure 2: Distribution of Time to Event (1999 Vintage)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 16)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig2_duration_hist.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f'\nDataset Summary:')
print(f'  N total    : {len(df):,}')
n_D = (df['event_code'] == 2).sum()
n_P = (df['event_code'] == 1).sum()
n_C = (df['event_code'] == 0).sum()
print(f'  n_D (default) : {n_D:,} ({n_D/len(df)*100:.1f}%)')
print(f'  n_P (prepay)  : {n_P:,} ({n_P/len(df)*100:.1f}%)')
print(f'  n_C (censored): {n_C:,} ({n_C/len(df)*100:.1f}%)')
print(f'\nDuration (years):')
print(f'  Mean  : {df["duration_years"].mean():.2f}')
print(f'  Median: {df["duration_years"].median():.2f}')
print(f'  Min   : {df["duration_years"].min():.2f}')
print(f'  Max   : {df["duration_years"].max():.2f}')

In [ ]:
# Covariate summary statistics
summary_stats = df[ALL_COVARIATES].describe().T
summary_stats['missing_%'] = df[ALL_COVARIATES].isna().mean() * 100
print('Covariate Summary Statistics:')
print(summary_stats[['mean', 'std', 'min', '50%', 'max', 'missing_%']].round(2).to_string())

---

## Section 6: Train/Test Split & Standardization

The paper trains on the full 1999 vintage. We use an 80/20 split so we
can evaluate out-of-sample predictive performance.

In [ ]:
# 80/20 stratified split
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.20, random_state=42,
    stratify=df['event_code'].values
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f'Train: {len(train_df):,}  Test: {len(test_df):,}')
for split_name, split_df in [('Train', train_df), ('Test', test_df)]:
    print(f'  {split_name}: default={int((split_df["event_code"]==2).sum()):,}, '
          f'prepay={int((split_df["event_code"]==1).sum()):,}, '
          f'censored={int((split_df["event_code"]==0).sum()):,}')

In [ ]:
# Standardize quantitative covariates (fit on train, transform both)
scaler = StandardScaler()

train_quant = scaler.fit_transform(train_df[QUANTITATIVE_COVARIATES].values)
test_quant = scaler.transform(test_df[QUANTITATIVE_COVARIATES].values)

# Combine with binary covariates and intercept
train_binary = train_df[BINARY_COVARIATES].values.astype(np.float64)
test_binary = test_df[BINARY_COVARIATES].values.astype(np.float64)

# Intercept column
train_intercept = np.ones((len(train_df), 1))
test_intercept = np.ones((len(test_df), 1))

# Full covariate matrices: [quantitative | binary | intercept]
X_train = np.hstack([train_quant, train_binary, train_intercept]).astype(np.float32)
X_test = np.hstack([test_quant, test_binary, test_intercept]).astype(np.float32)

FEATURE_NAMES = QUANTITATIVE_COVARIATES + BINARY_COVARIATES + ['intercept']
n_features = len(FEATURE_NAMES)

# Durations in years and events
duration_train = np.maximum(train_df['duration_years'].values.astype(np.float32), 1/24)
duration_test = np.maximum(test_df['duration_years'].values.astype(np.float32), 1/24)
event_train = train_df['event_code'].values.astype(np.int64)
event_test = test_df['event_code'].values.astype(np.int64)

print(f'X_train shape: {X_train.shape}')
print(f'Features ({n_features}): {FEATURE_NAMES}')
print(f'Duration range (years): [{duration_train.min():.3f}, {duration_train.max():.2f}]')

---

## Section 7: Model Specification

### Likelihood (Paper Eq. 4)

For loan $i$ with covariates $\mathbf{x}_i$, observed time $t_i$, and event
indicator $\delta_i \in \{0, D, P\}$:

$$L(\boldsymbol{\Theta}) = \prod_{i=1}^{N} \left[ \lambda_D(t_i | \mathbf{x}_i) \right]^{I(\delta_i = D)} \left[ \lambda_P(t_i | \mathbf{x}_i) \right]^{I(\delta_i = P)} S(t_i | \mathbf{x}_i)$$

where the overall survival function is:

$$S(t | \mathbf{x}) = \exp\left(-\Lambda_D(t | \mathbf{x}) - \Lambda_P(t | \mathbf{x})\right)$$

### Priors (Paper Section 3)

We use the paper’s priors exactly:
- $\mu_D, \mu_P \sim \mathcal{N}(0, 10^2)$ — **zero-mean** (diffuse)
- $\sigma_D, \sigma_P \sim \text{Exp}(0.01)$ — mean = 100
- $\boldsymbol{\theta}_D, \boldsymbol{\theta}_P \sim \mathcal{N}(\mathbf{0}, 100^2 \mathbf{I})$

Note: We define the Pyro model inline (rather than using the library class)
for full transparency and to match the paper’s zero-mean $\mu$ priors exactly.

In [ ]:
# --- Lognormal baseline hazard functions (numerically stable) ---
import math

_LOG_2PI = math.log(2 * math.pi)
_SQRT2 = math.sqrt(2)


def _log_standard_normal_pdf(z):
    """log phi(z) = -0.5*z^2 - 0.5*log(2*pi)"""
    return -0.5 * z**2 - 0.5 * _LOG_2PI


def _log_standard_normal_survival(z):
    """
    log(1 - Phi(z)) computed via erfc for numerical stability.
    
    1 - Phi(z) = 0.5 * erfc(z / sqrt(2))
    log(1 - Phi(z)) = log(0.5) + log(erfc(z / sqrt(2)))
    
    erfc is numerically stable for large z (unlike 1 - erf),
    and with float64 handles |z| up to ~26 before underflow.
    """
    return torch.log(torch.tensor(0.5, dtype=z.dtype, device=z.device)) + \
           torch.log(torch.erfc(z / _SQRT2).clamp(min=1e-30))


def lognormal_log_hazard(t, mu, sigma):
    """
    Log of lognormal hazard: log r(t | mu, sigma).
    
    log r(t) = log phi(z) - log(sigma) - log(t) - log(1 - Phi(z))
    """
    z = (torch.log(t) - mu) / sigma
    z = torch.clamp(z, -25, 25)
    log_phi = _log_standard_normal_pdf(z)
    log_surv = _log_standard_normal_survival(z)
    return log_phi - torch.log(sigma) - torch.log(t) - log_surv


def lognormal_cumulative_hazard(t, mu, sigma):
    """
    Cumulative lognormal baseline hazard: H_0(t) = -log(1 - Phi(z)).
    """
    z = (torch.log(t) - mu) / sigma
    z = torch.clamp(z, -25, 25)
    return -_log_standard_normal_survival(z)


print('Numerically stable lognormal hazard functions defined (using erfc).')

In [ ]:
# --- Prior hyperparameters (matching paper exactly) ---
PRIOR_THETA_SD = 100.0    # SD for regression coefficients
PRIOR_MU_SD = 10.0        # SD for baseline location (zero-mean per paper)
PRIOR_SIGMA_RATE = 0.01   # Rate for Exponential on sigma (mean=100)

# --- MCMC settings ---
NUM_WARMUP = 1000
NUM_SAMPLES = 5000
NUM_CHAINS = 4
TARGET_ACCEPT = 0.8


def bhattacharya_model(X, durations, events, n_features):
    """
    Pyro model for Bhattacharya et al. (2019).
    
    Key difference from library class: mu priors are N(0, 10^2)
    (zero-mean, matching paper) rather than N(3, 10).
    
    Uses float64 tensors throughout for numerical stability.
    """
    device = X.device
    dtype = X.dtype  # Inherit dtype from input (float64)

    # Baseline hazard parameters
    mu_D = pyro.sample('mu_D', dist.Normal(
        torch.tensor(0.0, dtype=dtype, device=device),
        torch.tensor(PRIOR_MU_SD, dtype=dtype, device=device)))
    sigma_D = pyro.sample('sigma_D', dist.Exponential(
        torch.tensor(PRIOR_SIGMA_RATE, dtype=dtype, device=device)))
    mu_P = pyro.sample('mu_P', dist.Normal(
        torch.tensor(0.0, dtype=dtype, device=device),
        torch.tensor(PRIOR_MU_SD, dtype=dtype, device=device)))
    sigma_P = pyro.sample('sigma_P', dist.Exponential(
        torch.tensor(PRIOR_SIGMA_RATE, dtype=dtype, device=device)))

    # Regression coefficients
    theta_D = pyro.sample(
        'theta_D',
        dist.Normal(torch.zeros(n_features, dtype=dtype, device=device),
                    PRIOR_THETA_SD * torch.ones(n_features, dtype=dtype, device=device)).to_event(1)
    )
    theta_P = pyro.sample(
        'theta_P',
        dist.Normal(torch.zeros(n_features, dtype=dtype, device=device),
                    PRIOR_THETA_SD * torch.ones(n_features, dtype=dtype, device=device)).to_event(1)
    )

    # Linear predictors
    eta_D = torch.matmul(X, theta_D)
    eta_P = torch.matmul(X, theta_P)

    # Cause-specific log-hazards
    log_h_D = lognormal_log_hazard(durations, mu_D, sigma_D) + eta_D
    log_h_P = lognormal_log_hazard(durations, mu_P, sigma_P) + eta_P

    # Cumulative hazards
    H_D = lognormal_cumulative_hazard(durations, mu_D, sigma_D) * torch.exp(eta_D)
    H_P = lognormal_cumulative_hazard(durations, mu_P, sigma_P) * torch.exp(eta_P)

    # Log-likelihood (Eq. 4)
    is_default = (events == 2).to(dtype)
    is_prepay = (events == 1).to(dtype)
    log_lik = is_default * log_h_D + is_prepay * log_h_P - H_D - H_P

    pyro.factor('log_likelihood', torch.sum(log_lik))


print(f'Model: {n_features} covariates (incl. intercept)')
print(f'Total parameters: {4 + 2 * n_features} (4 baseline + {2*n_features} coefficients)')
print(f'MCMC: {NUM_CHAINS} chains x {NUM_SAMPLES} samples ({NUM_WARMUP} warmup)')
print(f'Using float64 for numerical stability')

---

## Section 8: MCMC Inference

Run NUTS sampling. Multi-chain MCMC requires `mp_context='fork'` in
notebooks on macOS, or single-chain mode.

In [ ]:
%%time

pyro.clear_param_store()

# Convert data to float64 tensors — float32 lacks precision for
# the lognormal survival tail (erfc underflows around |z|>6 in float32,
# but handles |z|<26 in float64).
X_train_t = torch.tensor(X_train, dtype=torch.float64)
dur_train_t = torch.tensor(duration_train, dtype=torch.float64)
evt_train_t = torch.tensor(event_train, dtype=torch.int64)

# NUTS kernel
nuts = NUTS(
    bhattacharya_model,
    target_accept_prob=TARGET_ACCEPT,
    jit_compile=False,
)

# Run MCMC (single chain in notebook; for multi-chain, run as script)
mcmc = MCMC(
    nuts,
    num_samples=NUM_SAMPLES,
    warmup_steps=NUM_WARMUP,
    num_chains=1,  # Single chain for notebook compatibility
)

print(f'Running NUTS: 1 chain x {NUM_SAMPLES} samples ({NUM_WARMUP} warmup)...')
print(f'Tensor dtype: {X_train_t.dtype}')
mcmc.run(X_train_t, dur_train_t, evt_train_t, n_features)
print('MCMC completed.')

In [ ]:
# Extract posterior samples
posterior_samples = {k: v.cpu().numpy() for k, v in mcmc.get_samples().items()}
inference_data = az.from_pyro(mcmc)

print('Posterior sample shapes:')
for k, v in posterior_samples.items():
    print(f'  {k}: {v.shape}')

# Print MCMC summary (R-hat, ESS)
print('\n--- MCMC Summary ---')
mcmc.summary()

In [ ]:
# Convergence diagnostics
rhat = az.rhat(inference_data)
ess = az.ess(inference_data)

print('R-hat diagnostics (should be < 1.05):')
for var_name in ['mu_D', 'sigma_D', 'mu_P', 'sigma_P']:
    r = float(rhat[var_name].values)
    e = float(ess[var_name].values)
    status = 'OK' if r < 1.05 else 'WARNING'
    print(f'  {var_name:10s}: R-hat={r:.4f}, ESS={e:.0f}  [{status}]')

for param_name in ['theta_D', 'theta_P']:
    rhat_vals = rhat[param_name].values
    ess_vals = ess[param_name].values
    max_rhat = float(np.max(rhat_vals))
    min_ess = float(np.min(ess_vals))
    status = 'OK' if max_rhat < 1.05 else 'WARNING'
    print(f'  {param_name:10s}: max R-hat={max_rhat:.4f}, min ESS={min_ess:.0f}  [{status}]')

---

## Section 9: Posterior Analysis — Tables 1 & 2

**Table 1**: Baseline hazard parameter posteriors ($\mu_D, \sigma_D, \mu_P, \sigma_P$)

**Table 2**: Covariate coefficient posteriors ($\boldsymbol{\theta}_D, \boldsymbol{\theta}_P$)

In [ ]:
# Table 1: Baseline parameter posteriors
baseline_params = ['mu_D', 'sigma_D', 'mu_P', 'sigma_P']
table1_rows = []
for p in baseline_params:
    s = posterior_samples[p]
    table1_rows.append({
        'Parameter': p,
        'Median': np.median(s),
        'Mean': np.mean(s),
        'SD': np.std(s),
        'CI 2.5%': np.percentile(s, 2.5),
        'CI 97.5%': np.percentile(s, 97.5),
    })

table1 = pd.DataFrame(table1_rows)
print('Table 1: Posterior Summary of Baseline Hazard Parameters')
print('=' * 70)
print(table1.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Table 2: Covariate coefficient posteriors
def make_coef_table(param_name, label):
    rows = []
    samples = posterior_samples[param_name]
    for i, feat in enumerate(FEATURE_NAMES):
        s = samples[:, i]
        ci_lo, ci_hi = np.percentile(s, 2.5), np.percentile(s, 97.5)
        sig = '*' if not (ci_lo < 0 < ci_hi) else ''
        rows.append({
            'Covariate': feat,
            'Median': np.median(s),
            'Mean': np.mean(s),
            'SD': np.std(s),
            'CI 2.5%': ci_lo,
            'CI 97.5%': ci_hi,
            'Sig': sig,
        })
    return pd.DataFrame(rows)

table2_default = make_coef_table('theta_D', 'Default')
table2_prepay = make_coef_table('theta_P', 'Prepay')

print('Table 2a: Default Covariate Coefficients (theta_D)')
print('=' * 80)
print(table2_default.to_string(index=False, float_format='{:.4f}'.format))

print('\nTable 2b: Prepayment Covariate Coefficients (theta_P)')
print('=' * 80)
print(table2_prepay.to_string(index=False, float_format='{:.4f}'.format))

---

## Section 10: Posterior Density Plots — Figure 3

Six-panel posterior density plot for selected parameters, showing
combined posterior samples across chains.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 10))

# Credit score coefficients
cs_idx = FEATURE_NAMES.index('credit_score')
axes[0, 0].hist(posterior_samples['theta_D'][:, cs_idx], bins=60, density=True,
                alpha=0.7, color='#d0021b', edgecolor='white', linewidth=0.3)
axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0, 0].set_title('Credit Score (Default)', fontsize=11)
axes[0, 0].set_xlabel(r'$\theta_{D,\mathrm{credit\_score}}$')

axes[0, 1].hist(posterior_samples['theta_P'][:, cs_idx], bins=60, density=True,
                alpha=0.7, color='#4a90d9', edgecolor='white', linewidth=0.3)
axes[0, 1].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Credit Score (Prepayment)', fontsize=11)
axes[0, 1].set_xlabel(r'$\theta_{P,\mathrm{credit\_score}}$')

# Number of units coefficients
nu_idx = FEATURE_NAMES.index('num_units')
axes[1, 0].hist(posterior_samples['theta_D'][:, nu_idx], bins=60, density=True,
                alpha=0.7, color='#d0021b', edgecolor='white', linewidth=0.3)
axes[1, 0].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[1, 0].set_title('Number of Units (Default)', fontsize=11)
axes[1, 0].set_xlabel(r'$\theta_{D,\mathrm{num\_units}}$')

axes[1, 1].hist(posterior_samples['theta_P'][:, nu_idx], bins=60, density=True,
                alpha=0.7, color='#4a90d9', edgecolor='white', linewidth=0.3)
axes[1, 1].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Number of Units (Prepayment)', fontsize=11)
axes[1, 1].set_xlabel(r'$\theta_{P,\mathrm{num\_units}}$')

# Baseline mu parameters
axes[2, 0].hist(posterior_samples['mu_D'], bins=60, density=True,
                alpha=0.7, color='#d0021b', edgecolor='white', linewidth=0.3)
axes[2, 0].set_title(r'$\mu_D$ (Default Baseline)', fontsize=11)
axes[2, 0].set_xlabel(r'$\mu_D$')

axes[2, 1].hist(posterior_samples['mu_P'], bins=60, density=True,
                alpha=0.7, color='#4a90d9', edgecolor='white', linewidth=0.3)
axes[2, 1].set_title(r'$\mu_P$ (Prepayment Baseline)', fontsize=11)
axes[2, 1].set_xlabel(r'$\mu_P$')

for ax in axes.flat:
    ax.set_ylabel('Density')

fig.suptitle('Figure 3: Posterior Density Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig3_posteriors.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Section 11: Trace Plots — Figures 8, 9, 10

Trace plots for monitoring MCMC convergence and mixing.

In [ ]:
# Figure 8: Trace plots of baseline parameters
fig, axes = plt.subplots(4, 2, figsize=(14, 10))

for i, param in enumerate(baseline_params):
    samples = posterior_samples[param]
    # Trace
    axes[i, 0].plot(samples, alpha=0.6, linewidth=0.5, color='#4a90d9')
    axes[i, 0].set_ylabel(param)
    axes[i, 0].set_title(f'Trace: {param}', fontsize=10)
    if i == len(baseline_params) - 1:
        axes[i, 0].set_xlabel('Iteration')
    # Posterior
    axes[i, 1].hist(samples, bins=60, density=True, alpha=0.7, color='#4a90d9',
                    edgecolor='white', linewidth=0.3)
    axes[i, 1].axvline(np.median(samples), color='red', linestyle='--', linewidth=1.5,
                       label=f'median={np.median(samples):.3f}')
    axes[i, 1].legend(fontsize=8)
    axes[i, 1].set_title(f'Posterior: {param}', fontsize=10)

fig.suptitle('Figure 8: Baseline Parameter Traces', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig8_trace_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 9: Trace plots of default coefficients (theta_D)
n_coefs = n_features
n_cols = 3
n_rows = int(np.ceil(n_coefs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
axes_flat = axes.flatten()

for i in range(n_coefs):
    samples = posterior_samples['theta_D'][:, i]
    axes_flat[i].plot(samples, alpha=0.5, linewidth=0.4, color='#d0021b')
    axes_flat[i].set_title(f'{FEATURE_NAMES[i]}', fontsize=9)
    axes_flat[i].axhline(0, color='black', linestyle='--', alpha=0.3)

# Hide unused axes
for j in range(n_coefs, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(r'Figure 9: Trace Plots of Default Coefficients ($\theta_D$)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig9_trace_default.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 10: Trace plots of prepayment coefficients (theta_P)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
axes_flat = axes.flatten()

for i in range(n_coefs):
    samples = posterior_samples['theta_P'][:, i]
    axes_flat[i].plot(samples, alpha=0.5, linewidth=0.4, color='#4a90d9')
    axes_flat[i].set_title(f'{FEATURE_NAMES[i]}', fontsize=9)
    axes_flat[i].axhline(0, color='black', linestyle='--', alpha=0.3)

for j in range(n_coefs, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(r'Figure 10: Trace Plots of Prepayment Coefficients ($\theta_P$)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig10_trace_prepay.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Section 12: Posterior Predictive Distributions — Figure 4, Table 3

Select 3 default and 3 prepaid mortgages, show their covariate values
(Table 3) and plot posterior predictive density of time to event (Figure 4).

In [ ]:
# Select 3 default and 3 prepaid mortgages from test set
np.random.seed(123)

default_idx = np.where(event_test == 2)[0]
prepay_idx = np.where(event_test == 1)[0]

if len(default_idx) >= 3:
    selected_default = np.random.choice(default_idx, 3, replace=False)
else:
    selected_default = default_idx[:min(3, len(default_idx))]

selected_prepay = np.random.choice(prepay_idx, 3, replace=False)

# Table 3: Covariate values for selected mortgages
table3_rows = []
for label, indices in [('Default', selected_default), ('Prepaid', selected_prepay)]:
    for j, idx in enumerate(indices):
        row = {'Type': label, 'Mortgage': f'{label[0]}{j+1}',
               't_obs (yr)': f'{duration_test[idx]:.2f}'}
        for k, feat in enumerate(FEATURE_NAMES):
            row[feat] = f'{X_test[idx, k]:.2f}'
        table3_rows.append(row)

table3 = pd.DataFrame(table3_rows)
print('Table 3: Covariate Values for Selected Mortgages')
print('=' * 100)
print(table3.to_string(index=False))

In [ ]:
# Figure 4: Posterior predictive distributions of time to event

def compute_posterior_predictive_density(X_i, cause, t_grid, posterior_samples):
    """
    Compute posterior predictive density f(t) = lambda(t) * S(t)
    for a single observation across posterior samples.
    """
    X_i_t = torch.tensor(X_i, dtype=torch.float32)
    t_t = torch.tensor(t_grid, dtype=torch.float32)
    
    mu_key = 'mu_D' if cause == 'default' else 'mu_P'
    sigma_key = 'sigma_D' if cause == 'default' else 'sigma_P'
    theta_key = 'theta_D' if cause == 'default' else 'theta_P'
    
    n_post = len(posterior_samples[mu_key])
    n_t = len(t_grid)
    density_samples = np.zeros((n_post, n_t))
    
    for s in range(n_post):
        mu = torch.tensor(posterior_samples[mu_key][s], dtype=torch.float32)
        sigma = torch.tensor(posterior_samples[sigma_key][s], dtype=torch.float32)
        theta = torch.tensor(posterior_samples[theta_key][s], dtype=torch.float32)
        
        eta = torch.dot(X_i_t, theta)
        
        # Cause-specific hazard
        log_h = lognormal_log_hazard(t_t, mu, sigma) + eta
        
        # Overall survival (both risks)
        mu_D_s = torch.tensor(posterior_samples['mu_D'][s], dtype=torch.float32)
        sigma_D_s = torch.tensor(posterior_samples['sigma_D'][s], dtype=torch.float32)
        mu_P_s = torch.tensor(posterior_samples['mu_P'][s], dtype=torch.float32)
        sigma_P_s = torch.tensor(posterior_samples['sigma_P'][s], dtype=torch.float32)
        theta_D_s = torch.tensor(posterior_samples['theta_D'][s], dtype=torch.float32)
        theta_P_s = torch.tensor(posterior_samples['theta_P'][s], dtype=torch.float32)
        
        eta_D = torch.dot(X_i_t, theta_D_s)
        eta_P = torch.dot(X_i_t, theta_P_s)
        
        H_D = lognormal_cumulative_hazard(t_t, mu_D_s, sigma_D_s) * torch.exp(eta_D)
        H_P = lognormal_cumulative_hazard(t_t, mu_P_s, sigma_P_s) * torch.exp(eta_P)
        S_t = torch.exp(-H_D - H_P)
        
        # f(t) = lambda_k(t) * S(t)
        density_samples[s] = (torch.exp(log_h) * S_t).numpy()
    
    return np.mean(density_samples, axis=0), np.percentile(density_samples, 2.5, axis=0), \
           np.percentile(density_samples, 97.5, axis=0)


t_grid = np.linspace(0.1, 15, 200)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Default mortgages
colors_d = ['#d0021b', '#e8553a', '#f4a460']
for j, idx in enumerate(selected_default):
    mean_d, lo_d, hi_d = compute_posterior_predictive_density(
        X_test[idx], 'default', t_grid, posterior_samples)
    axes[0].plot(t_grid, mean_d, color=colors_d[j], label=f'D{j+1} (t={duration_test[idx]:.1f}yr)')
    axes[0].fill_between(t_grid, lo_d, hi_d, alpha=0.15, color=colors_d[j])
    axes[0].axvline(duration_test[idx], color=colors_d[j], linestyle=':', alpha=0.7)

axes[0].set_title('Posterior Predictive: Time to Default', fontsize=12)
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Prepayment mortgages
colors_p = ['#4a90d9', '#6bb5e8', '#a0d2f0']
for j, idx in enumerate(selected_prepay):
    mean_p, lo_p, hi_p = compute_posterior_predictive_density(
        X_test[idx], 'prepay', t_grid, posterior_samples)
    axes[1].plot(t_grid, mean_p, color=colors_p[j], label=f'P{j+1} (t={duration_test[idx]:.1f}yr)')
    axes[1].fill_between(t_grid, lo_p, hi_p, alpha=0.15, color=colors_p[j])
    axes[1].axvline(duration_test[idx], color=colors_p[j], linestyle=':', alpha=0.7)

axes[1].set_title('Posterior Predictive: Time to Prepayment', fontsize=12)
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Density')
axes[1].legend()

fig.suptitle('Figure 4: Posterior Predictive Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig4_predictive.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Section 13: Model Assessment — Figures 5, 6, 7

Following the paper’s assessment methodology:
- **Standardized residuals**: $(t_\mathrm{obs} - E[T]) / \mathrm{sd}(T)$
- **Posterior predictive reliability**: $R(t_\mathrm{obs})$ at observed event times
- **Coverage**: Fraction of observed times within 95% prediction intervals

In [ ]:
# Model wrapper compatible with bayesian_evaluation module
class BayesianModelWrapper:
    def __init__(self, posterior_samples, device='cpu'):
        self.posterior_samples_ = posterior_samples
        self.device = device

    def predict_cif(self, X, times, cause='default'):
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        times_t = torch.tensor(times, dtype=torch.float32, device=self.device)
        N, T = X_t.shape[0], len(times_t)

        ps = {k: torch.tensor(v, device=self.device) for k, v in self.posterior_samples_.items()}
        n_samples = len(ps['mu_D'])
        cif_samples = np.zeros((n_samples, N, T))

        for s in range(n_samples):
            eta_D = torch.matmul(X_t, ps['theta_D'][s])
            eta_P = torch.matmul(X_t, ps['theta_P'][s])
            for t_idx, t in enumerate(times_t):
                H_D = lognormal_cumulative_hazard(t, ps['mu_D'][s], ps['sigma_D'][s]) * torch.exp(eta_D)
                H_P = lognormal_cumulative_hazard(t, ps['mu_P'][s], ps['sigma_P'][s]) * torch.exp(eta_P)
                S_t = torch.exp(-H_D - H_P)
                total_H = H_D + H_P + 1e-10
                if cause == 'default':
                    cif_samples[s, :, t_idx] = ((H_D / total_H) * (1 - S_t)).cpu().numpy()
                else:
                    cif_samples[s, :, t_idx] = ((H_P / total_H) * (1 - S_t)).cpu().numpy()

        return np.mean(cif_samples, 0), np.percentile(cif_samples, 2.5, 0), np.percentile(cif_samples, 97.5, 0)

    def predict_survival(self, X, times):
        X_t = torch.tensor(X, dtype=torch.float32, device=self.device)
        times_t = torch.tensor(times, dtype=torch.float32, device=self.device)
        N, T = X_t.shape[0], len(times_t)

        ps = {k: torch.tensor(v, device=self.device) for k, v in self.posterior_samples_.items()}
        n_samples = len(ps['mu_D'])
        surv_samples = np.zeros((n_samples, N, T))

        for s in range(n_samples):
            eta_D = torch.matmul(X_t, ps['theta_D'][s])
            eta_P = torch.matmul(X_t, ps['theta_P'][s])
            for t_idx, t in enumerate(times_t):
                H_D = lognormal_cumulative_hazard(t, ps['mu_D'][s], ps['sigma_D'][s]) * torch.exp(eta_D)
                H_P = lognormal_cumulative_hazard(t, ps['mu_P'][s], ps['sigma_P'][s]) * torch.exp(eta_P)
                surv_samples[s, :, t_idx] = torch.exp(-H_D - H_P).cpu().numpy()

        return np.mean(surv_samples, 0), np.percentile(surv_samples, 2.5, 0), np.percentile(surv_samples, 97.5, 0)

model_wrapper = BayesianModelWrapper(posterior_samples)
print('Model wrapper created for evaluation.')

In [ ]:
# Compute standardized residuals for default and prepaid mortgages
from competing_risks.bayesian_evaluation import compute_standardized_residuals

resid_default, t_obs_default = compute_standardized_residuals(
    model_wrapper, X_test, duration_test, event_test, cause='default'
)
resid_prepay, t_obs_prepay = compute_standardized_residuals(
    model_wrapper, X_test, duration_test, event_test, cause='prepay'
)

print(f'Default residuals: n={len(resid_default)}, mean={np.mean(resid_default):.3f}, std={np.std(resid_default):.3f}')
print(f'Prepay residuals:  n={len(resid_prepay)}, mean={np.mean(resid_prepay):.3f}, std={np.std(resid_prepay):.3f}')

In [ ]:
# Figure 5: Combined box plot of all standardized residuals
all_resid = np.concatenate([resid_default, resid_prepay])
all_labels = ['Default'] * len(resid_default) + ['Prepay'] * len(resid_prepay)

fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot([all_resid], labels=['All Events'], widths=0.5,
           patch_artist=True, boxprops=dict(facecolor='#e8e8e8'))
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Standardized Residual')
ax.set_title('Figure 5: Standardized Residuals (All Events)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig5_residuals_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Separate box plots for default vs. prepaid residuals
fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot(
    [resid_default, resid_prepay],
    labels=['Default', 'Prepaid'],
    widths=0.5,
    patch_artist=True,
)
bp['boxes'][0].set_facecolor('#f4a4a4')
bp['boxes'][1].set_facecolor('#a4c8f4')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Standardized Residual')
ax.set_title('Figure 6: Standardized Residuals by Event Type', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig6_residuals_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Box plots of posterior predictive reliability at observed event times

def compute_reliability_at_event(X_events, t_events, posterior_samples):
    """Compute S(t_obs) for each event observation across posterior samples."""
    X_t = torch.tensor(X_events, dtype=torch.float32)
    n_obs = len(t_events)
    ps = {k: torch.tensor(v) for k, v in posterior_samples.items()}
    n_post = len(ps['mu_D'])
    
    reliabilities = np.zeros((n_post, n_obs))
    for s in range(n_post):
        eta_D = torch.matmul(X_t, ps['theta_D'][s])
        eta_P = torch.matmul(X_t, ps['theta_P'][s])
        for i in range(n_obs):
            t_i = torch.tensor(t_events[i], dtype=torch.float32)
            H_D = lognormal_cumulative_hazard(t_i, ps['mu_D'][s], ps['sigma_D'][s]) * torch.exp(eta_D[i])
            H_P = lognormal_cumulative_hazard(t_i, ps['mu_P'][s], ps['sigma_P'][s]) * torch.exp(eta_P[i])
            reliabilities[s, i] = torch.exp(-H_D - H_P).item()
    
    return np.mean(reliabilities, axis=0)  # Posterior mean reliability per observation

# Default reliabilities
default_mask = event_test == 2
prepay_mask = event_test == 1

if default_mask.sum() > 0:
    rel_default = compute_reliability_at_event(
        X_test[default_mask], duration_test[default_mask], posterior_samples)
else:
    rel_default = np.array([])

rel_prepay = compute_reliability_at_event(
    X_test[prepay_mask], duration_test[prepay_mask], posterior_samples)

fig, ax = plt.subplots(figsize=(7, 5))
data_to_plot = []
labels_to_plot = []
if len(rel_default) > 0:
    data_to_plot.append(rel_default)
    labels_to_plot.append(f'Default\n(n={len(rel_default)})')
data_to_plot.append(rel_prepay)
labels_to_plot.append(f'Prepaid\n(n={len(rel_prepay)})')

bp = ax.boxplot(data_to_plot, labels=labels_to_plot, widths=0.5, patch_artist=True)
colors_fig7 = ['#f4a4a4', '#a4c8f4'] if len(rel_default) > 0 else ['#a4c8f4']
for patch, color in zip(bp['boxes'], colors_fig7):
    patch.set_facecolor(color)

ax.set_ylabel('Reliability $R(t_{obs})$')
ax.set_title('Figure 7: Posterior Predictive Reliability at Observed Event Times', fontsize=12)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_fig7_reliability.png', dpi=150, bbox_inches='tight')
plt.show()

# Coverage analysis
print('\nReliability Summary:')
if len(rel_default) > 0:
    print(f'  Default: mean R(t_obs)={np.mean(rel_default):.3f}, median={np.median(rel_default):.3f}')
print(f'  Prepaid: mean R(t_obs)={np.mean(rel_prepay):.3f}, median={np.median(rel_prepay):.3f}')

---

## Section 14: ML Comparison — Tables 4 & 5

Following the paper, compare the Bayesian model against:
- **Logistic Regression + Lasso** (multinomial)
- **Random Forest** classifier

Predict the category: default / prepaid / active.

In [ ]:
# Prepare targets for classification (3-class: default=2, prepaid=1, active=0)
y_train_cls = event_train.copy()
y_test_cls = event_test.copy()

# Use the covariates WITHOUT intercept for ML models
X_train_ml = X_train[:, :-1]  # Drop intercept column
X_test_ml = X_test[:, :-1]

label_names = {0: 'Active', 1: 'Prepaid', 2: 'Default'}

print(f'ML classification data:')
print(f'  Train: {X_train_ml.shape}, Test: {X_test_ml.shape}')
for code in [0, 1, 2]:
    print(f'  {label_names[code]}: train={int((y_train_cls==code).sum()):,}, test={int((y_test_cls==code).sum()):,}')

In [ ]:
# Table 4: Logistic Regression + Lasso (multinomial)
lr = LogisticRegression(
    penalty='l1', solver='saga', multi_class='multinomial',
    max_iter=5000, C=1.0, random_state=42,
)
lr.fit(X_train_ml, y_train_cls)
y_pred_lr = lr.predict(X_test_ml)

cm_lr = confusion_matrix(y_test_cls, y_pred_lr, labels=[0, 1, 2])
cm_lr_df = pd.DataFrame(
    cm_lr,
    index=[f'True {label_names[i]}' for i in [0, 1, 2]],
    columns=[f'Pred {label_names[i]}' for i in [0, 1, 2]],
)

print('Table 4: Confusion Matrix — Logistic Regression + Lasso')
print('=' * 55)
print(cm_lr_df.to_string())
print(f'\nAccuracy: {(y_pred_lr == y_test_cls).mean():.4f}')
print('\nClassification Report:')
print(classification_report(y_test_cls, y_pred_lr, target_names=['Active', 'Prepaid', 'Default']))

In [ ]:
# Table 5: Random Forest classifier
rf = RandomForestClassifier(
    n_estimators=500, max_depth=10, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1,
)
rf.fit(X_train_ml, y_train_cls)
y_pred_rf = rf.predict(X_test_ml)

cm_rf = confusion_matrix(y_test_cls, y_pred_rf, labels=[0, 1, 2])
cm_rf_df = pd.DataFrame(
    cm_rf,
    index=[f'True {label_names[i]}' for i in [0, 1, 2]],
    columns=[f'Pred {label_names[i]}' for i in [0, 1, 2]],
)

print('Table 5: Confusion Matrix — Random Forest')
print('=' * 55)
print(cm_rf_df.to_string())
print(f'\nAccuracy: {(y_pred_rf == y_test_cls).mean():.4f}')
print('\nClassification Report:')
print(classification_report(y_test_cls, y_pred_rf, target_names=['Active', 'Prepaid', 'Default']))

In [ ]:
# Visual comparison of confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cm, title in [(axes[0], cm_lr, 'Logistic Regression + Lasso'),
                       (axes[1], cm_rf, 'Random Forest')]:
    # Normalize by row
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(['Active', 'Prepaid', 'Default'])
    ax.set_yticklabels(['Active', 'Prepaid', 'Default'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=11)

    for i in range(3):
        for j in range(3):
            color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.0%})',
                    ha='center', va='center', color=color, fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.8, label='Proportion')
fig.suptitle('ML Comparison: Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bhattacharya_ml_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Section 15: Conclusion & Save Artifacts

In [ ]:
# Save posterior samples
np.savez(
    MODELS_DIR / 'bhattacharya_posterior_samples.npz',
    **posterior_samples,
)

# Save inference data (ArviZ NetCDF)
inference_data.to_netcdf(str(MODELS_DIR / 'bhattacharya_inference_data.nc'))

# Save coefficient tables
table1.to_csv(MODELS_DIR / 'bhattacharya_table1_baseline.csv', index=False)
table2_default.to_csv(MODELS_DIR / 'bhattacharya_table2_default.csv', index=False)
table2_prepay.to_csv(MODELS_DIR / 'bhattacharya_table2_prepay.csv', index=False)

# Save feature names and scaler
import pickle
with open(MODELS_DIR / 'bhattacharya_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(MODELS_DIR / 'bhattacharya_features.pkl', 'wb') as f:
    pickle.dump(FEATURE_NAMES, f)

print('Artifacts saved to', MODELS_DIR)

In [ ]:
print('=' * 70)
print('BHATTACHARYA ET AL. (2019) REPLICATION — SUMMARY')
print('=' * 70)

print(f'\nData: 1999 Vintage (Freddie Mac sample)')
print(f'  Total loans: {len(df):,}')
print(f'  Defaults: {n_D:,} ({n_D/len(df)*100:.1f}%)')
print(f'  Prepayments: {n_P:,} ({n_P/len(df)*100:.1f}%)')
print(f'  Censored: {n_C:,} ({n_C/len(df)*100:.1f}%)')

print(f'\nModel: Bayesian Competing Risks PHM (Lognormal Baseline)')
print(f'  Covariates: {n_features} (incl. intercept)')
print(f'  MCMC: NUTS, 1 chain x {NUM_SAMPLES} samples ({NUM_WARMUP} warmup)')

print(f'\nTable 1 — Baseline Parameters (Posterior Median [95% CI]):')
for _, row in table1.iterrows():
    print(f'  {row["Parameter"]:8s}: {row["Median"]:8.4f}  [{row["CI 2.5%"]:8.4f}, {row["CI 97.5%"]:8.4f}]')

print(f'\nTable 2 — Significant Covariates (95% CI excludes 0):')
sig_D = table2_default[table2_default['Sig'] == '*']['Covariate'].tolist()
sig_P = table2_prepay[table2_prepay['Sig'] == '*']['Covariate'].tolist()
print(f'  Default:  {", ".join(sig_D) if sig_D else "None"}')
print(f'  Prepay:   {", ".join(sig_P) if sig_P else "None"}')

print(f'\nML Comparison (Test Accuracy):')
print(f'  Logistic + Lasso: {(y_pred_lr == y_test_cls).mean():.4f}')
print(f'  Random Forest:    {(y_pred_rf == y_test_cls).mean():.4f}')

print(f'\nFigures saved to: {FIGURES_DIR}')
print(f'Model artifacts saved to: {MODELS_DIR}')
print('=' * 70)